# 06 — Radar Duman Testi (F3 ADIM 5)

`mode='live'` ile tüm evrende 3 şablon. Hedef: **< 10 ms** (performans
bütçesi §15). Dönen sonuçların grafikleri de render edilir — canlı modda
gelenin makul görünüp görünmediği erken anlaşılsın.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
OUT = Path("notebooks/output"); OUT.mkdir(parents=True, exist_ok=True)
TF = "1d"

import time
from engine.search.engine import SearchEngine
from engine.templates import get
from tools.f3_measure import render_top12

motor = SearchEngine()
sablonlar = ["head_and_shoulders", "cup_and_handle", "bull_flag"]

# Isınma (memmap sayfa cache)
motor.search(get(sablonlar[0]).vector, mode="live", timeframe=TF)

olcum = []
for ad in sablonlar:
    q = get(ad).vector
    t0 = time.perf_counter()
    r = motor.search(q, mode="live", timeframe=TF, tolerance="flexible", limit=12)
    ms = (time.perf_counter() - t0) * 1000
    olcum.append(dict(sablon=ad, sure_ms=round(ms, 2), sonuc=r.total,
                      en_iyi=round(r.results[0].distance, 4) if r.results else None))
pd.DataFrame(olcum)

,sablon,sure_ms,sonuc,en_iyi
0,head_and_shoulders,5.64,21,0.0600
1,cup_and_handle,5.55,15,0.1084
2,bull_flag,5.48,14,0.1137


In [2]:
# Radar sonuçları gerçekten SON pencere mi?
r = motor.search(get("cup_and_handle").vector, mode="live", timeframe=TF, tolerance="flexible", limit=12)
for m in r.results[:6]:
    h = motor.handle(TF, m.window_length)
    son = int(h.end_idx[h.symbol_id == m.symbol_id].max())
    print(f"{m.symbol:11s} L={m.window_length:3d} end_idx={m.end_idx:5d} sembolün son penceresi={son:5d} "
          f"{'OK' if m.end_idx == son else 'HATA'}")

ETHUSDT     L= 64 end_idx= 3267 sembolün son penceresi= 3267 OK
PEPEUSDT    L= 40 end_idx= 1179 sembolün son penceresi= 1179 OK
RNDRUSDT    L= 24 end_idx=  968 sembolün son penceresi=  968 OK
ZECUSDT     L= 40 end_idx= 2685 sembolün son penceresi= 2685 OK
BTCUSDT     L= 40 end_idx= 3267 sembolün son penceresi= 3267 OK
FDUSDUSDT   L= 24 end_idx= 1097 sembolün son penceresi= 1097 OK


In [3]:
yol, sonuc = render_top12(motor, get("cup_and_handle").vector, "radar_cup_and_handle",
                          TF, OUT, tolerance="flexible", mode="live")
print("render:", yol)
print("radar sonuç sayısı:", sonuc.total)

render: notebooks\output\goz_testi_radar_cup_and_handle.png
radar sonuç sayısı: 15
